In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb # or import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import metrics
import lightgbm as lgb

In [2]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
print("Columns in train DataFrame:", df_train.columns.tolist())
print("DataFrame shape:", df_train.shape)
print("Columns in test DataFrame:", df_test.columns.tolist())
print("DataFrame shape:", df_test.shape)

Columns in train DataFrame: ['Index', 'geohash', 'day', 'timestamp', 'demand', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']
DataFrame shape: (77299, 11)
Columns in test DataFrame: ['Index', 'geohash', 'day', 'timestamp', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather']
DataFrame shape: (41778, 10)


In [3]:
df_train['is_test'] = 0
df_test['is_test'] = 1
df_test['demand'] = np.nan
df_all = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
df_all = df_all.sort_values(by=['geohash', 'day', 'timestamp']).reset_index(drop=True)

In [4]:
geohash_means = df_all.groupby('geohash')['demand'].mean()
df_all['geohash_encoded'] = df_all['geohash'].map(geohash_means)


In [5]:
if 'Temperature' in df_all.columns:
    df_all['Temperature'] = df_all['Temperature'].fillna(df_all['Temperature'].median())


categorical_cols = ['RoadType', 'LargeVehicles', 'Weather', 'Landmarks']
for col in categorical_cols:
    if col in df_all.columns:
        df_all[col] = df_all[col].fillna('Unknown')


print("Missing values per column:\n", df_all.isnull().sum())

Missing values per column:
 Index                  0
geohash                0
day                    0
timestamp              0
demand             41778
RoadType               0
NumberofLanes          0
LargeVehicles          0
Landmarks              0
Temperature            0
Weather                0
is_test                0
geohash_encoded       25
dtype: int64


In [6]:
df_all[['Hour', 'Minute']] = df_all['timestamp'].str.split(':', expand=True).astype(int)
df_all['TimeInMinutes'] = df_all['Hour'] * 60 + df_all['Minute']
df_all['sin_time'] = np.sin(2 * np.pi * df_all['TimeInMinutes'] / 1440.0)
df_all['cos_time'] = np.cos(2 * np.pi * df_all['TimeInMinutes'] / 1440.0)

In [ ]:

df_all['demand_lag_1'] = df_all.groupby('geohash')['demand'].shift(1)
df_all['demand_lag_2'] = df_all.groupby('geohash')['demand'].shift(2)

train_median = df_train['demand'].median()
df_all['demand_lag_1'] = df_all['demand_lag_1'].fillna(train_median)
df_all['demand_lag_2'] = df_all['demand_lag_2'].fillna(train_median)

df_all['demand_momentum'] = df_all['demand_lag_1'] - df_all['demand_lag_2']

df_all['geohash_encoded'] = df_all['geohash_encoded'].fillna(train_median)

In [ ]:
columns_to_encode = ['RoadType', 'LargeVehicles']

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_matrix = encoder.fit_transform(df_all[columns_to_encode])
encoded_df = pd.DataFrame(encoded_matrix, columns=encoder.get_feature_names_out(columns_to_encode))


df_all = df_all.reset_index(drop=True)
df_all = pd.concat([df_all, encoded_df], axis=1)
df_all = df_all.drop(columns=columns_to_encode)

In [9]:
for col in ['Weather', 'Landmarks']:
    if col in df_all.columns:
        df_all[col] = df_all[col].astype('category').cat.codes

In [10]:
print("Missing values per column:\n", df_all.isnull().sum())

Missing values per column:
 Index                            0
geohash                          0
day                              0
timestamp                        0
demand                       41778
NumberofLanes                    0
Landmarks                        0
Temperature                      0
Weather                          0
is_test                          0
geohash_encoded                  0
Hour                             0
Minute                           0
TimeInMinutes                    0
sin_time                         0
cos_time                         0
demand_lag_1                     0
demand_lag_2                     0
demand_momentum                  0
RoadType_Highway                 0
RoadType_Residential             0
RoadType_Street                  0
RoadType_Unknown                 0
LargeVehicles_Allowed            0
LargeVehicles_Not Allowed        0
dtype: int64


In [11]:
train_final = df_all[df_all['is_test'] == 0].copy()
test_final = df_all[df_all['is_test'] == 1].copy()

drop_cols = ['demand', 'timestamp', 'geohash', 'Index', 'is_test']
X_train = train_final.drop(columns=drop_cols, errors='ignore')
y_train = train_final['demand']
X_test = test_final.drop(columns=drop_cols, errors='ignore')

In [12]:
final_model = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=7,
    # objective='reg:pseudohubererror',
    subsample=0.85, 
    colsample_bytree=0.85,  
    colsample_bylevel=0.7, 
    random_state=42,
    n_jobs=-1     
)
final_model.fit(X_train, y_train)

# lgb_model = lgb.LGBMRegressor(
#     n_estimators=600,
#     learning_rate=0.03,       # Slower learning rate for precision
#     num_leaves=31,            # CRITICAL: Caps the maximum leaves per tree (default is often too high)
#     max_depth=6,              # CRITICAL: Forces a hard ceiling on vertical growth
#     min_child_samples=20,     # Prevents creating leaves that only cover a few specific rows/geohashes
#     subsample=0.8,            # Row bagging
#     colsample_bytree=0.8,     # Feature bagging
#     random_state=42,
#     n_jobs=-1,
#     verbose=-1
# )
# lgb_model.fit(X_train, y_train)





print("Final model trained successfully!")

Final model trained successfully!


In [13]:
test_final['predicted_demand'] = final_model.predict(X_test)
# test_final['predicted_demand'] = lgb_model.predict(X_test)
# xgb_preds = final_model.predict(X_test)  # Your 73-score model
# lgb_preds = lgb_model.predict(X_test)

# test_final['predicted_demand'] = (0.75 * xgb_preds) + (0.25 * lgb_preds)

In [14]:
submission = pd.DataFrame({
    'Index': test_final['Index'],
    'demand': test_final['predicted_demand']
})

submission = submission.sort_values(by='Index').reset_index(drop=True)

# Save to CSV
submission.to_csv('prediction3.csv', index=False)
print("Submission file 'prediction.csv' generated and saved successfully!")

Submission file 'prediction.csv' generated and saved successfully!
